In [4]:
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="Car Price Predictor", layout="wide")
st.title("Предсказание цены автомобиля")

# Загружаем лучшую модель
@st.cache_resource
def load_objects():
    with open('best_model.pkl','rb') as f:
        objects=pickle.load(f)
    return (objects['best_model'],objects['ohe'],objects['feature_columns'],objects['num_cols'],objects['cat_cols'])
best_model,ohe,feature_columns,num_cols,cat_cols=load_objects()
cat_cols = [c for c in cat_cols if c != 'seats']
def predict_price(input_df):
    X_cat = ohe.transform(input_df[cat_cols])
    X_cat_df = pd.DataFrame(X_cat, columns=ohe.get_feature_names_out(cat_cols))
    X_num = input_df[num_cols].reset_index(drop=True)
    X_num['year_squared'] = X_num['year']**2
    X_num['power_per_engine'] = X_num['max_power']/(X_num['engine']+1)
    X_num['hp_per_l'] = X_num['max_power']*1000/(X_num['engine']+1)
    X_final = pd.concat([X_num, X_cat_df], axis=1)

    # Отладка — покажет проблемные колонки
    missing_cols = set(feature_columns) - set(X_final.columns)
    extra_cols = set(X_final.columns) - set(feature_columns)
    if missing_cols:
        st.error(f"Отсутствуют колонки: {missing_cols}")
    if extra_cols:
        st.error(f"Лишние колонки: {extra_cols}")

    X_final = X_final.reindex(columns=feature_columns, fill_value=0)

    return np.expm1(best_model.predict(X_final))
# Боковая панель
menu=st.sidebar.selectbox(
    "Выбор режима",
    ("EDA","Предсказание","Веса"))

if menu=="EDA":
    st.header("Разведочный анализ данных")
    uploaded=st.file_uploader("Загрузите csv-файл",type=['csv'])
    if uploaded:
        df=pd.read_csv(uploaded)
        st.success(f"Загружено {len(df)} строк")

        if 'selling_price' in df.columns:
            st.subheader("Распределение цены")
            fig,ax=plt.subplots()
            df['selling_price'].hist(bins=50,edgecolor='black',ax=ax)
            ax.set_xlabel('Цена')
            st.pyplot(fig)

        if 'max_power' in df.columns and 'selling_price' in df.columns:
            st.subheader("Зависимость цены от мощности")
            fig,ax=plt.subplots()
            ax.scatter(df['max_power'],df['selling_price'],alpha=0.5)
            ax.set_xlabel('Мощность')
            ax.set_ylabel('Цена')
            st.pyplot(fig)

        if 'year' in df.columns and 'selling_price' in df.columns:
            st.subheader("Зависимость цены от года выпуска")
            fig,ax=plt.subplots()
            ax.scatter(df['year'],df['selling_price'],alpha=0.5)
            ax.set_xlabel('Год')
            ax.set_ylabel('Цена')
            st.pyplot(fig)

        numeric=df.select_dtypes(include=[np.number]).columns
        if len(numeric)>1:
            st.subheader("Корреляция числовых признаков")
            fig,ax=plt.subplots(figsize=(10,6))
            sns.heatmap(df[numeric].corr(),annot=True,cmap='coolwarm',ax=ax)
            st.pyplot(fig)

elif menu == "Предсказание":
    st.header("Предсказание цены автомобиля")

    input_type=st.radio("Способ ввода:",["Вручную","Загрузить CSV"])

    if input_type=="Вручную":
        with st.form("manual_form"):
            col1,col2=st.columns(2)
            with col1:
                year=st.number_input("Год",1983,2020,2015)
                km=st.number_input("Пробег",0,500000,50000)
                fuel=st.selectbox("Топливо",['CNG','Diesel','LPG','Petrol'])
                seller=st.selectbox("Продавец",['Dealer','Individual','Trustmark Dealer'])
                brand=st.selectbox("Бренд",['Ambassador', 'Audi', 'BMW', 'Chevrolet', 'Daewoo', 'Datsun', 'Fiat', 'Force', 'Ford', 'Honda', 'Hyundai', 'Isuzu', 'Jaguar', 'Jeep', 'Kia', 'Land', 'Lexus', 'MG', 'Mahindra', 'Maruti', 'Mercedes-Benz', 'Mitsubishi', 'Nissan', 'Peugeot', 'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo'])
                model=st.text_input("Модель","Swift")
            with col2:
                trans=st.selectbox("КПП",['Automatic','Manual'])
                owner=st.selectbox("Владелец",['First Owner','Fourth & Above Owner','Second Owner','Test Drive Car','Third Owner'])
                mileage=st.number_input("Расход",0.0,50.0,20.0)
                engine=st.number_input("Объём",500,5000,1500)
                power=st.number_input("Мощность",0.0,500.0,100.0)
                torque=st.number_input("Момент",0.0,2000.0,150.0)
                max_rpm=st.number_input("Обороты макс. момента",0,10000,4000)
                seats=st.selectbox("Количество мест",sorted([2,4,5,6,7,8,9,10,14]))

            submitted=st.form_submit_button("Рассчитать цену")
            if submitted:
                input_df=pd.DataFrame([{
                    'year':year,'km_driven':km,'mileage':mileage,
                    'engine':engine,'max_power':power,'torque':torque,
                    'max_torque_rpm':max_rpm,'brand':brand,'model':model,
                    'fuel':fuel,'seller_type':seller,'transmission':trans,
                    'owner':owner,'seats':seats
                }])

                try:
                    price=predict_price(input_df)[0]
                    st.success(f"Предсказанная цена: {price:,.0f} ₽")
                except Exception as e:
                    st.error(f"Ошибка: {e}")

    else:
        uploaded=st.file_uploader("Загрузите csv с признаками",type=['csv'])
        if uploaded:
            df=pd.read_csv(uploaded)
            st.write("Данные:",df.head())
            if st.button("Расчет"):
                try:
                    preds=predict_price(df)
                    df['predicted_price']=preds
                    st.dataframe(df[['predicted_price']])
                    csv=df.to_csv(index=False).encode('utf-8')
                    st.download_button("Скачать результат",csv,"predictions.csv")
                except Exception as e:
                    st.error(f"Ошибка: {e}")

elif menu == "Веса":
    st.header("Важность признаков")

    coef=best_model.coef_
    imp=pd.DataFrame({'Признак':feature_columns,'Вес':coef})
    imp=imp.sort_values('Вес',key=lambda x:abs(x),ascending=False)

    fig,ax = plt.subplots(figsize=(10,6))
    top=imp.head(5)
    colors=['red' if w<0 else 'green' for w in top['Вес']]
    ax.barh(top['Признак'],top['Вес'],color=colors)
    ax.axvline(0,color='black')
    ax.set_xlabel('Коэффициент')
    st.pyplot(fig)
    st.caption("зелёный - увеличивает цену, красный - уменьшает цену")
    with st.expander("Показать все веса"):
        st.dataframe(imp)


-rw-r--r-- 1 root root 115317 May 10 16:53 model.pkl
⠙⠹⠸⠼⠴⠦⠧⠇⠏

2026-05-10 17:03:33.411 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.171.48.202:8501

your url is: https://late-beans-stare.loca.lt
  Stopping...
^C
